# Constraint-Based Modelling of Rhodopseudomonas palustris

Ejemplo de Optimización con el Modelo de R. palustris (iDT1294) from Tec-Campos et al., 2023 en COBRApy

Objetivo:
Maximizar la tasa de crecimiento y producción de PHA de R. palustris.

Pasos:
Instalación y carga del modelo: El modelo básico de R. palustris (iDT1294) from Tec-Campos et al., 2023.

## *Installation

In [1]:
# Install cobrapy
!pip install -qq cobra escher networkx memote catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.0/961.0 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.2/342.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 

In [2]:
# Check installation
!pip show cobra

Name: cobra
Version: 0.30.0
Summary: COBRApy is a package for constraint-based modeling of metabolic networks.
Home-page: https://opencobra.github.io/cobrapy
Author: The cobrapy core development team.
Author-email: cobra-pie@googlegroups.com
License: LGPL-2.0-or-later OR GPL-2.0-or-later
Location: /usr/local/lib/python3.12/dist-packages
Requires: appdirs, depinfo, diskcache, future, httpx, numpy, optlang, pandas, pydantic, python-libsbml, rich, ruamel.yaml, swiglpk
Required-by: Escher, memote


## *Mount drive

In [3]:
# Mount drive
from google.colab import drive
import os
import sys
import pandas as pd
from pathlib import Path


drive_folder = "Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/models"
def mount_drive():
  drive.mount('/content/driveDL', force_remount=True)
  os.chdir("/content/driveDL/My Drive/"+drive_folder)
mount_drive()

# Project root = current working directory
project_root = Path(os.getcwd())

Mounted at /content/driveDL


## *Load packages

In [4]:
# load packages

import logging

import cobra
from cobra import Reaction, Metabolite, Model, io
from cobra.io import (
    load_model, load_json_model, save_json_model,
    load_matlab_model, save_matlab_model,
    read_sbml_model, write_sbml_model
)
from cobra.util.solver import linear_reaction_coefficients

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import seaborn as sns
import json
import os
import math
import copy
from itertools import product
from tqdm import tqdm


In [5]:
!python --version

Python 3.12.12


## *Load model

In [6]:
print(project_root)


/content/driveDL/My Drive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/models


In [7]:
# Load models

# Directory where the model lives
phbv_model_dir = project_root / "phbv"
phbv_model_dir.mkdir(parents=True, exist_ok=True)

# Full path to the SBML file
phbv_model_path = phbv_model_dir / "model_rpalustris_PHBV.xml"

# Load model
model_phbv = io.read_sbml_model(
    str(phbv_model_path),
    use_fbc=False
)

print("Model loaded successfully")

Model loaded successfully


# *Check model with PHBV reactions based on tec_campos_iDT1294

In [8]:
# Load model
model_phbv

Name,iDT1294
Memory address,7aa93e9dabd0
Number of metabolites,2128
Number of reactions,2739
Number of genes,1294
Number of groups,118
Objective expression,1.0*PHBVS_syn - 1.0*PHBVS_syn_reverse_aec1c
Compartments,"c, u, p, e"


In [9]:
# Check model summary
model_phbv.summary() # PHBVS_syn is the new objective function

Metabolite,Reaction,Flux,C-Number,C-Flux
actn__R_e,EX_actn__R_e,0.8502,4,79.08%
cinnm_e,EX_cinnm_e,0.09995,9,20.92%
o2_e,EX_o2_e,0.1999,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
h2_c,DM_h2_c,-0.5001,0,0.00%
h2o_e,EX_h2o_e,-0.0002,0,0.00%
hco3_e,EX_hco3_e,-0.09985,1,99.60%
PHBV_c,SK_phbv_c,-1,0,0.00%
2obut_c,sink_2obut_c,-0.0001,4,0.40%


In [10]:
# Check model compartments
model_phbv.compartments

{'c': 'c', 'u': 'u', 'p': 'p', 'e': 'e'}

# *PHBV reactions


In [12]:
# Check reaction directly associated to PHB production
phbv_rxn = model_phbv.reactions.get_by_id("PHBVS_syn")
print(phbv_rxn.bounds)
print(phbv_rxn.reaction)


(0.0, 1.0)
0.8 3hbcoa__R_c + 0.2 3hvcoa__R_c --> PHBV_c + coa_c


## *Copy model

In [13]:
# Copy phbv model
model_phbv2 = model_phbv.copy()

## Opt: Change PHBVS_syn bounds

In [ ]:
# Specify PHBVS_syn reaction bound

rxn = model_phbv2.reactions.get_by_id("PHBVS_syn")
rxn.lower_bound = 0
rxn.upper_bound = 1
phbv_rxn = model_phbv2.reactions.get_by_id("PHBVS_syn")
print(phbv_rxn.bounds)
print(phbv_rxn.reaction)


(0, 0.5)
0.8 3hbcoa__R_c + 0.2 3hvcoa__R_c --> PHBV_c + coa_c


## *Opt: save reactions and metabolites for reference

In [ ]:
# Define output directory
phbv_model_dir = project_root / "phbv" / "checkpoint_output"
phbv_model_dir.mkdir(parents=True, exist_ok=True)

# -----------------------
# Save reactions table
# -----------------------
rxn_data = []

for r in model_phbv2.reactions:
    rxn_data.append({
        "id": r.id,
        "name": r.name,
        "equation": r.reaction,
        "lower_bound": r.lower_bound,
        "upper_bound": r.upper_bound,
        "genes": ";".join(g.id for g in r.genes),
    })

rxn_df = pd.DataFrame(rxn_data)
rxn_df.to_csv(phbv_model_dir / "model_PHBV_reactions.csv", index=False)

# -----------------------
# Save metabolites table
# -----------------------
met_data = []

for m in model_phbv2.metabolites:
    met_data.append({
        "id": m.id,
        "name": m.name,
        "formula": m.formula,
        "compartment": m.compartment,
        "charge": m.charge,
        "in_reactions": len(m.reactions),
    })

met_df = pd.DataFrame(met_data)
met_df.to_csv(phbv_model_dir / "model_PHBV_metabolites.csv", index=False)


## *Opt: save default medium of PHBV model

Genome scale-metabolic models simulate media through the reactions staring with "EX" with negative flow.

In [ ]:
# Check default medium conditions
model_phbv2.medium

# Convert to a DataFrame
medium_df = pd.DataFrame(list(model_phbv2.medium.items()), columns=["Reaction", "UptakeBound"])

# Save to Excel
medium_df.to_csv(phbv_model_dir/ "PHBV_model_default_medium.xlsx", index=False)

# Print the medium elements
#for rxn_id, bound in model.medium.items():
#   print(rxn_id, bound)



# *Medium Montiel-Corona 2022

To simulate experiemntal medium from Montiel-Corona et al., Montiel-Corona 2022

Note:
Photon reactions are not connected by default.
Wavelengths >700 nm missing



## *Montiel-Corona 2022 with conservative upper bounds

In [14]:
# Set medium as specified in Montiel-Corona & Buitrón 2022.

# === CONSERVATIVE UPPER-BOUND UPTAKE LIMITS (from medium composition) ===
# These are maximal possible average uptake rates (mmol·gDW⁻¹·h⁻¹)
# computed from the medium composition, assuming full consumption over 72 h.
# They are safe, conservative constraints.

nh4_from_acetate = 0.06590  # ammonium from NH4-acetate
nh4_from_nh4cl = 0.07598    # ammonium from NH4Cl
total_nh4_uptake = nh4_from_acetate + nh4_from_nh4cl

medium_montiel_buitron_2022_conservative_upperbound = {
    "EX_ac_e":     0.2477,        # acetate (but we override with q_ac =0.1209 later)
    # Cofeeding
    #"EX_fum_e":    0,       # test: fumarate increases production a lot
    #"EX_hxa_e":    0,       # test: hexanoate
    #"EX_ppa_e":    0,       # test: propionate does not help much
    # Other nutrients
    "EX_nh4_e":   total_nh4_uptake,    # ammonium (from NH4-acetate & NH4Cl)
    "EX_pi_e":     0.03733,       # phosphate (KH2PO4)
    "EX_mg2_e":    0.01604,       # magnesium (MgSO4)
    "EX_na1_e":    0.06955,       # sodium (NaCl)
    "EX_ca2_e":    0.003455,      # calcium
    "EX_fe3_e":    0.0004147,     # iron(III)
    #"EX_cys__L_e": 0.01933,       # L-cysteine·HCl
    #"EX_thm_e":    0.00003363,    # thiamine·HCl
    #"EX_nac_e":    0.00008250,    # nicotinic acid
    "EX_zn2_e":    0.000002337,   # Zn
    "EX_mn2_e":    0.000000620,   # Mn
    "EX_bo3_e":    0.00001972,    # Boron
    #"EX_co2pp_e":  0.000003414,   # Cobalt (ID guessed)
    "EX_cu2_e":    0.000000234,   # Copper
    "EX_ni2_e":    0.000000345,   # Nickel
    "EX_mobd_e":   0.000000559,   # molybdate (NaMoO4)
    "EX_co2_e":    0.0,          # CO2
    "EX_o2_e":     0.0,          # set to 0 if anaerobic; set to >0 if aerobic
    #"EX_hco3_e":   0.0,          # bicarbonate

    # photons for photosynthesis are not connected yet. This is why setting the to zero does not change much.
    'EX_photon410_e': 100,
    'EX_photon430_e': 100,
    'EX_photon450_e': 100,
    'EX_photon470_e': 100,
    'EX_photon490_e': 100,
    'EX_photon510_e': 100,
    'EX_photon530_e': 100,
    'EX_photon550_e': 100,
    'EX_photon570_e': 100,
    'EX_photon590_e': 100,
    'EX_photon610_e': 100,
    'EX_photon630_e': 100,
    'EX_photon650_e': 100,
    'EX_photon670_e': 100,
    'EX_photon690_e': 100
}

# Close ALL uptake (sets a clean minimal medium)
for ex in model_phbv2.exchanges:
    ex.lower_bound = 0.0

# Apply medium_montiel_buitron_2022_conservative_upperbound
print("\n=== Applying conservative medium upper-bound uptake limits ===")
for rxn_id, max_uptake in medium_montiel_buitron_2022_conservative_upperbound.items():
    if rxn_id in model_phbv2.reactions:
        model_phbv2.reactions.get_by_id(rxn_id).lower_bound = -max_uptake
        print(f"{rxn_id}: lower_bound = {-max_uptake:.6g}")
    else:
        print(f"{rxn_id}: NOT FOUND IN MODEL (skipped)")



=== Applying conservative medium upper-bound uptake limits ===
EX_ac_e: lower_bound = -0.2477
EX_nh4_e: lower_bound = -0.14188
EX_pi_e: lower_bound = -0.03733
EX_mg2_e: lower_bound = -0.01604
EX_na1_e: lower_bound = -0.06955
EX_ca2_e: lower_bound = -0.003455
EX_fe3_e: lower_bound = -0.0004147
EX_zn2_e: lower_bound = -2.337e-06
EX_mn2_e: lower_bound = -6.2e-07
EX_bo3_e: lower_bound = -1.972e-05
EX_cu2_e: lower_bound = -2.34e-07
EX_ni2_e: lower_bound = -3.45e-07
EX_mobd_e: lower_bound = -5.59e-07
EX_co2_e: lower_bound = -0
EX_o2_e: lower_bound = -0
EX_photon410_e: lower_bound = -100
EX_photon430_e: lower_bound = -100
EX_photon450_e: lower_bound = -100
EX_photon470_e: lower_bound = -100
EX_photon490_e: lower_bound = -100
EX_photon510_e: lower_bound = -100
EX_photon530_e: lower_bound = -100
EX_photon550_e: lower_bound = -100
EX_photon570_e: lower_bound = -100
EX_photon590_e: lower_bound = -100
EX_photon610_e: lower_bound = -100
EX_photon630_e: lower_bound = -100
EX_photon650_e: lower_boun

## Opt: Tweak uptake reactions

In [ ]:

# Acetate upper/lower bounds
#acetate_exchange = model_acetate1.reactions.get_by_id('EX_ac_e')
#acetate_exchange.lower_bound = -10.0  # Set the lower bound (uptake)
#acetate_exchange.upper_bound = 1000.0  # Set the upper boun (secretion)

# Hexanoate upper/lower bounds
#hexanoate_exchange = model_acetate1.reactions.get_by_id('EX_hxa_e')
#hexanoate_exchange.lower_bound = -10.0  # Set the lower bound (uptake)
#hexanoate_exchange.upper_bound = 1000.0  # Set the upper bound (secretion)

# Butyrate upper/lower bounds
#butyrate_exchange = model_acetate1.reactions.get_by_id('EX_but_e')
#butyrate_exchange.lower_bound = 0.0  # Set the lower bound (uptake)
#butyrate_exchange.upper_bound = 0.0  # Set the upper bound (secretion)

# Check medium changes
#for exchange, flux in model_acetate1.medium.items():
#    print(exchange, flux)


### Opt: Alernative Electron Sinks

In [ ]:
# Enable alternative electron carbon sinks to test electron leakage to other products

#h2_exchange = model_acetate1.reactions.get_by_id('EX_h2_e') # H2 is the second electron sink of purple bacteria
#h2_exchange.lower_bound = 0  # Set the lower bound (uptake)
#h2_exchange.upper_bound = 10.0  # Set the upper boun (secretion)

#no3_exchange = model_acetate1.reactions.get_by_id('EX_no3_e')
#no3_exchange.lower_bound = -1000.0  # Set the lower bound (uptake)
#no3_exchange.upper_bound = 0  # Set the upper boun (secretion)


## *Checkpoint: Save medium medium_montiel_buitron_2022 with exchange bounds

In [ ]:
# Save medium medium_montiel_buitron_2022_conservative_upperbound

# Collect bounds for all exchange reactions
rows = []
for rxn in model_phbv2.exchanges:
    rows.append({
        "Reaction": rxn.id,
        "Name": rxn.name,
        "LowerBound": rxn.lower_bound,
        "UpperBound": rxn.upper_bound
    })

# Convert to DataFrame
medium_df = pd.DataFrame(rows)

# Save to Excel
medium_df.to_csv(phbv_model_dir / "PHBV_model_medium_montiel_buitron_2022_exchange_bounds.xlsx", index=False)

medium_df.head()


,Reaction,Name,LowerBound,UpperBound
0,EX_h2o_e,H2O exchange,0.0,1000.0
1,EX_o2_e,O2 exchange,-0.0,1000.0
2,EX_co2_e,CO2 exchange,-0.0,1000.0
3,EX_leu__L_e,L-Leucine exchange,0.0,0.0
4,EX_cobalt2_e,Co2+ exchange,0.0,1000.0


### Opt: check biomass reaction bounds

In [15]:
# Check reaction EX_ac_e
rxn = model_phbv2.reactions.get_by_id('BIOMASS__1') #PDH
print(f"{rxn.id}: lower = {rxn.lower_bound}, upper = {rxn.upper_bound}") # bounds

BIOMASS__1: lower = 0.0, upper = 2.0


### Opt: Constrain biomass reaction

To constrain the model further.

In [ ]:
# Only run if model needs further constaints

model_phbv2.reactions.BIOMASS__1.lower_bound = 0.001626
model_phbv2.reactions.BIOMASS__1.upper_bound = 1


### Opt: set reactions to zero

To investigate which reactions are involved incertain pathways (e.g., PHA production)

In [ ]:
# Tokens to search for (lowercase)
cofactor_tokens = {
    'ATP': ['atp'],
    'ADP': ['adp'],
    'AMP': ['amp'],
    'Pi': ['pi', 'phosphate'],
    'H': ['h', 'proton'],
    'NAD': ['nad(', 'nad_c', 'nad_', 'nicotinamide adenine dinucleotide'],
    'NADH': ['nadh', 'nad_h', 'nadh_c'],
    'NADP': ['nadp', 'nadp_c'],
    'NADPH': ['nadph', 'nadph_c'],
    'FAD': ['fad'],
    'FADH2': ['fadh2', 'fad_h2'],
    'Q': ['q', 'quinone', 'ubiquinone', 'menaquinone'],
    'QH2': ['qh2', 'quinol', 'ubiquinol', 'menaquinol']
}

# Build a map: cofactor -> list of metabolite objects
cofactor_mets = {k: [] for k in cofactor_tokens}

for met in model_phbv2.metabolites:
    mid = (met.id or "").lower()
    mname = (met.name or "").lower()
    for cof, toks in cofactor_tokens.items():
        for t in toks:
            if t in mid or t in mname:
                cofactor_mets[cof].append(met)
                break

# Collect all unique reactions involving these metabolites
cofactor_reactions_set = set()
for mets in cofactor_mets.values():
    for met in mets:
        for rxn in met.reactions:
            cofactor_reactions_set.add(rxn)

# Set bounds of these reactions to zero
for rxn in cofactor_reactions_set:
    rxn.lower_bound = 0
    rxn.upper_bound = 0

print(f"Set flux bounds to zero for {len(cofactor_reactions_set)} reactions associated with cofactors.")


Set flux bounds to zero for 2415 reactions associated with cofactors.


### Opt: Find reactions are associated to certain pathway

In [ ]:
# Identify and diable reactions in the "Fatty acid biosynthesis" subsystem
fatty_acid_reactions = [rxn for rxn in model_phbv2.reactions if rxn.subsystem and 'Fatty acid biosynthesis' in rxn.subsystem]
print(fatty_acid_reactions)
# Disable these reactions by setting bounds to zero
for rxn in fatty_acid_reactions:
    rxn.lower_bound = 0
    rxn.upper_bound = 0

print(f"Disabled {len(fatty_acid_reactions)} reactions in Fatty acid biosynthesis.")


[<Reaction 3HAD40_2 at 0x784ff8205010>, <Reaction 3OAR40_2 at 0x784ff82057c0>, <Reaction ALDR18 at 0x784ff8207d70>, <Reaction ALDDC17 at 0x784ff8207e30>, <Reaction DASYN181_9 at 0x784ff82ca0c0>, <Reaction DASYN182_9_12 at 0x784ff82ca180>, <Reaction DASYN183_6_9_12 at 0x784ff82ca1b0>, <Reaction DASYN183_9_12_15 at 0x784ff82ca1e0>, <Reaction DASYN184_6_9_12_15 at 0x784ff82ca210>, <Reaction G3PAT181_9 at 0x784ff82ca480>, <Reaction G3PAT182_9_12 at 0x784ff82ca510>, <Reaction G3PAT183_6_9_12 at 0x784ff82ca540>, <Reaction G3PAT183_9_12_15 at 0x784ff82ca570>, <Reaction G3PAT184_6_9_12_15 at 0x784ff82ca5a0>, <Reaction PGSA181_9 at 0x784ff82cb410>, <Reaction PGSA182_9_12 at 0x784ff82cb440>, <Reaction PGSA183_6_9_12 at 0x784ff82cb470>, <Reaction PGSA183_9_12_15 at 0x784ff82cb4a0>, <Reaction PGSA184_6_9_12_15 at 0x784ff82cb4d0>]
Disabled 19 reactions in Fatty acid biosynthesis.


## *Checkpoint: Save all reactions

To snapshot reactions before performing optimization


In [16]:
# Check reaction EX_ac_e bounds before optimization
rxn = model_phbv2.reactions.get_by_id('DM_PHBV_c') #PDH #PHBS_syn
print(f"{rxn.id}: lower = {rxn.lower_bound}, upper = {rxn.upper_bound}") # bounds

DM_PHBV_c: lower = 0.0, upper = 1000.0


In [ ]:
# Collect all reactions in a list of dicts
rxn_data = []
for rxn in model_phbv2.reactions:
    rxn_data.append({
        "ReactionID": rxn.id,
        "Name": rxn.name,
        "Equation": rxn.build_reaction_string(use_metabolite_names=True),
        "LowerBound": rxn.lower_bound,
        "UpperBound": rxn.upper_bound,
        "ObjectiveCoeff": rxn.objective_coefficient,
        "Subsystem": rxn.subsystem
    })

# Convert to DataFrame
rxn_df = pd.DataFrame(rxn_data)

# Save to Excel
rxn_df.to_csv(phbv_model_dir / "PHBV_model_all_reactions_bounds.xlsx", index=False)

print("Saved complete reaction table to model_all_reaction_bounds.xlsx")


Saved complete reaction table to model_all_reaction_bounds.xlsx


# *Save model phbv

In [17]:
# Save model with phbvv reactions in phbvv directory
phbv_model_dir = project_root / "phbv"

# Ensure folder exists
phbv_model_dir.mkdir(parents=True, exist_ok=True)

# Target file
phbv_model_file = phbv_model_dir / "model_rpalustris_PHBV_constrained.xml"

# Save COBRA model as .xml file
cobra.io.write_sbml_model(model_phbv, str(phbv_model_file))

